# Lab 01 — Time Series Feature Engineering

本 lab 面向剛開始接觸 AIOps、網路監控資料與 Jupyter 的學員。\
我們從原始監控計數器出發，把它整理成後續異常偵測可直接使用的特徵，並存成 `outputs/self-study/features.csv`。

```
原始計數器欄位
  → 合併 IN / OUT 為總量特徵
  → 計算比例特徵（封包錯誤率、丟棄率）
  → 用 rolling statistics 建立短期 baseline
  → 儲存 features.csv 供 Lab 02–7 使用
```

**你會用到的欄位**

| 欄位 | 初學者解讀 |
|---|---|
| `INOCTETS` / `OUTOCTETS` | 進入 / 離開介面的 bytes 數；可粗略理解成流量大小 |
| `INUCASTPKTS` / `OUTUCASTPKTS` | 單播封包數；多數正常主機通訊都屬於這類 |
| `INERRORS` / `OUTERRORS` | 有錯誤的封包數；常和線路或硬體問題有關 |
| `INDISCARDS` / `OUTDISCARDS` | 被設備丟棄的封包數；常和壅塞或 queue 滿有關 |
| `INBROADCASTPKTS` / `OUTBROADCASTPKTS` | broadcast 封包數；異常升高時可能是 broadcast storm |
| `INMULTICASTPKTS` / `OUTMULTICASTPKTS` | multicast 封包數 |
| `INUNKNOWNPROTOS` | 設備無法辨識協定的封包數；可作為異常線索 |


In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab01_feature_pipeline.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab01_feature_pipeline.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab01_feature_pipeline.svg")


## Lab 01 概覽

### 學習目標

1. 理解原始計數器（counter）與語意特徵（feature）的差異
2. 計算三類特徵：總量、比例、對稱性
3. 掌握 rolling 視窗 的直觀意義與實作方式
4. 視覺化原始值 vs rolling baseline，辨識異常時段的輪廓

### 前置條件

Lab 00 完成（exporter 執行中，`data/synthetic/synthetic_rrd_metrics.csv` 存在）

## 設計主線：特徵決定演算法看得見什麼

本章的實務連結是把設備計數器翻譯成維運語言。`INOCTETS` 和 `OUTOCTETS` 本身只是累積值；`traffic_total`、`error_rate`、`broadcast_ratio`、rolling baseline 才能回答「現在是否偏離正常」。

設計特徵時請問三個問題：

1. **語意是否清楚**：這個 feature 對應到哪種實務現象，例如壅塞、線路錯誤、broadcast storm、流量驟降？
2. **比較是否公平**：不同 port 的 baseline 不同，是否需要 rate、ratio 或 per-port rolling window？
3. **後續方法是否吃得到**：Z-score 需要穩定 baseline，SPC 需要正常期，ML 需要同一時間窗的多維 feature vector。

設計原則：不要先問要用哪個模型。先問要讓模型看見哪個維運現象。


## 欄位參考：每個數字代表什麼？

在計算特徵前，先理解每個計數器的物理意義。這不是可選的——
調錯欄位的方向（例如把「丟棄」誤判為「流量」）會讓整個偵測邏輯失效。

| 類別 | 欄位 | 意義 | 異常時常見原因 |
|---|---|---|---|
| 流量 | `INOCTETS` / `OUTOCTETS` | 進 / 出介面的 bytes 數 | 備份、串流、業務流量變化 |
| 單播封包 | `INUCASTPKTS` / `OUTUCASTPKTS` | 一對一封包數 | 連線數增加、服務請求增加 |
| 非單播 | `INNUCASTPKTS` / `OUTNUCASTPKTS` | Broadcast + Multicast 合計（舊 MIB） | ARP/DHCP 風暴、multicast flooding |
| 錯誤 | `INERRORS` / `OUTERRORS` | 有誤封包數 | 線材、SFP、NIC、duplex mismatch |
| 丟棄 | `INDISCARDS` / `OUTDISCARDS` | 設備主動丟棄的封包 | Queue 滿、壅塞、QoS 策略 |
| Broadcast | `INBROADCASTPKTS` / `OUTBROADCASTPKTS` | Broadcast 封包數（新 MIB） | ARP storm、switch loop |
| Multicast | `INMULTICASTPKTS` / `OUTMULTICASTPKTS` | Multicast 封包數 | IGMP 問題、IPTV、routing 異常 |
| 未知協定 | `INUNKNOWNPROTOS` | 設備無法辨識的協定 | 非標準封包、設定錯誤 |

**關鍵區別**：
- `ERRORS` → 傳輸失敗（硬體 / 線路層問題）
- `DISCARDS` → 傳輸成功但被設備丟棄（軟體 / 資源層問題）

混淆這兩類會導致告警方向錯誤。



In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

# ── 找專案根目錄（無論從哪個工作目錄啟動 Jupyter 都能運作）
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "outputs" / "self-study"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# ── 載入原始資料
df = pd.read_csv(DATA_SYNTHETIC / "synthetic_rrd_metrics.csv",
                 parse_dates=["timestamp"])
events = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv",
                     parse_dates=["start_time", "end_time"])
df = df.sort_values(["device_id", "port_id", "timestamp"]).reset_index(drop=True)

print(f"資料筆數：{len(df):,}")
print(f"Port 數量：{df.port_id.nunique()}  |  "
      f"Time範圍：{df.timestamp.min().date()} → {df.timestamp.max().date()}")
display(df.head(3))


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 1 — 先看事件表（Ground Truth）

`events` 是本課程的 ground truth，即「我們已知哪些Time真的發生了事件」。\
真實工作環境通常沒有這份資料，但這裡用它來驗證後面的偵測方法準不準。


In [ ]:
display(events)
print()
print("事件類型分布：")
print(events.event_type.value_counts().to_string())


---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 5 分鐘，請先不要執行 cell。

---

## 📖 概念：為什麼原始計數器不夠？什麼是特徵？

### 原始計數器的問題

Prometheus 收到的 `INOCTETS` 是一個**累積計數器（cumulative counter）**：
它只會遞增，每次都是「從設備啟動到current共傳輸了多少 bytes」。

直接用這個數字偵測異常毫無意義，因為：
- 今天下午 3 點的值是 `8,450,000,000`，明天下午 3 點可能是 `8,520,000,000`——
  差值才是流量，絕對值什麼都說不了
- 不同 port 的流量規模差十倍，直接比較沒有意義

**計數器必須先轉換，才能偵測異常。**

### 什麼是特徵（feature）？

特徵是你從原始資料**設計出來**的、對偵測有意義的數值。
在監控脈絡下，一個好的特徵應該：

1. 規模不變性——不因流量大小不同而讓小 port 的異常被掩蓋
2. 語意清晰——`error_rate`（錯誤率）比 `INERRORS`（絕對錯誤數）更容易判讀
3. 包含時序資訊——「current比過去高多少」比「current是多少」更能捕捉異常

### 三類特徵的直觀意義

| 類型 | 範例 | 直觀意義 |
|---|---|---|
| 總量 | `traffic_total = INOCTETS + OUTOCTETS` | 雙向流量合計，去掉方向噪音 |
| 比例 | `error_rate = INERRORS / INUCASTPKTS` | 每 100 個封包有幾個出錯，規模無關 |
| 對稱性 | `traffic_asymmetry = (IN-OUT)/(IN+OUT)` | +1 = 全進、-1 = 全出，偵測方向異常 |

**特徵是工程師的決策，不是演算法的輸出。**
演算法只能計算你告訴它的特徵；如果你沒有設計 `broadcast_total`，
演算法永遠不會注意到 broadcast storm。

## Step 2 — 把原始計數器轉成語意特徵

原始欄位通常太細，初學者很難直接判讀。\
例如 `INOCTETS` 和 `OUTOCTETS` 分開看時只能各自判斷；合成 `traffic_total` 後，
才能一眼看出整體流量的趨勢。

這裡計算三類特徵：
1. **總量（total）** — IN + OUT 合併，讓後續分析不用同時追蹤兩個方向
2. **比例（rate）** — 錯誤數 / 封包數，過濾流量規模的影響
3. **對稱性（asymmetry）** — (IN − OUT) / (IN + OUT)，偵測單向流量異常


In [ ]:
def safe_div(a, b):
    """Safe division that avoids division by zero."""
    return np.where(b > 0, a / b, 0.0)

# 總量特徵
df["traffic_total"]   = df["INOCTETS"]       + df["OUTOCTETS"]
df["ucast_total"]     = df["INUCASTPKTS"]    + df["OUTUCASTPKTS"]
df["errors_total"]    = df["INERRORS"]        + df["OUTERRORS"]
df["discards_total"]  = df["INDISCARDS"]      + df["OUTDISCARDS"]
df["broadcast_total"] = df["INBROADCASTPKTS"] + df["OUTBROADCASTPKTS"]
df["multicast_total"] = df["INMULTICASTPKTS"] + df["OUTMULTICASTPKTS"]

# Ratio特徵
df["error_rate"]   = safe_div(df["errors_total"],   df["ucast_total"])
df["discard_rate"] = safe_div(df["discards_total"],  df["ucast_total"])
df["bcast_rate"]      = safe_div(df["broadcast_total"], df["ucast_total"])
df["broadcast_ratio"] = df["bcast_rate"]
df["multicast_ratio"] = safe_div(df["multicast_total"], df["ucast_total"])
df["unknown_total"]   = df["INUNKNOWNPROTOS"]

# 平均封包大小（octets / packets）— 大封包通常是備份或大檔傳輸
df["avg_packet_size"] = safe_div(df["traffic_total"], df["ucast_total"])

# Traffic對稱性：-1 全出 / 0 對稱 / +1 全入
df["in_out_asymmetry"] = safe_div(
    df["INOCTETS"] - df["OUTOCTETS"],
    df["INOCTETS"] + df["OUTOCTETS"]
)

feature_cols = [c for c in df.columns if c not in
    ["INOCTETS","OUTOCTETS","INUCASTPKTS","OUTUCASTPKTS",
     "INERRORS","OUTERRORS","INDISCARDS","OUTDISCARDS",
     "INBROADCASTPKTS","OUTBROADCASTPKTS","INMULTICASTPKTS","OUTMULTICASTPKTS",
     "INUNKNOWNPROTOS"]]
print("衍生特徵：", [c for c in df.columns if c not in
    ["timestamp","device_id","port_id","port_role","event_label",
     "INOCTETS","OUTOCTETS","INUCASTPKTS","OUTUCASTPKTS","INERRORS","OUTERRORS",
     "INDISCARDS","OUTDISCARDS","INBROADCASTPKTS","OUTBROADCASTPKTS",
     "INMULTICASTPKTS","OUTMULTICASTPKTS","INUNKNOWNPROTOS"]])


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3 — 先選一個 port 畫三張圖

在看整個資料集之前，先固定看 `port-id7427`（Firewall 的第一個 port），
因為它包含 `queue_congestion` 事件，可以同時示範流量、封包大小與丟棄率的變化。

紅色陰影 = 已知事件時段（ground truth）。


In [ ]:
PORT = "port-id7427"
p = df[df["port_id"] == PORT].copy().set_index("timestamp")

def shade_events(ax, port_id, events):
    for _, ev in events[events.port_id == port_id].iterrows():
        ax.axvspan(ev.start_time, ev.end_time, alpha=0.15, color="crimson", zorder=0)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Panel 1 — Traffic
axes[0].plot(p.index, p["traffic_total"], color="steelblue", lw=1, label="traffic_total")
axes[0].set_ylabel("Bytes / 5 min")
axes[0].set_title(f"{PORT} — Traffic, packet size, error/discard rate")
shade_events(axes[0], PORT, events)

# Panel 2 — 平均封包大小
axes[1].plot(p.index, p["avg_packet_size"], color="darkorange", lw=1, label="avg_packet_size")
axes[1].set_ylabel("Bytes / packet")
shade_events(axes[1], PORT, events)

# Panel 3 — 錯誤率 vs 丟棄率
axes[2].plot(p.index, p["error_rate"],   color="tomato",      lw=1, label="error_rate")
axes[2].plot(p.index, p["discard_rate"], color="mediumpurple", lw=1, label="discard_rate")
axes[2].set_ylabel("Rate (errors / ucast)")
axes[2].legend(fontsize=9)
shade_events(axes[2], PORT, events)

for ax in axes:
    ax.legend(fontsize=9, loc="upper left")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 5 分鐘，請先不要執行 cell。

---

## 📖 概念：Rolling 視窗 作為「移動記憶」

### 直觀類比

想像你是一位值班工程師，你不可能記住所有時間點的數值，
但你知道「最近一個小時這個 port 的流量大概在 1–2 MB/min」——
這個印象就是 rolling mean。

Rolling 視窗 讓演算法也有同樣的「短期記憶」：
每個時間點只往回看固定長度的窗口，計算那段Time的統計量。

```
Time軸 →
  t-11  t-10  t-9  ...  t-1   t（current）
  ──────────────────────────────────────
  └──────── 往回看 12 個點（1 小時）────┘
                                        ↑
              rolling mean = 這 12 個點的平均
              rolling std  = 這 12 個點的波動幅度

  下一個時間點 t+1：
  t-10  t-9  ...  t-1   t   t+1（current）
  └──────── 窗口右移一格 ─────────────────┘
```

### 兩種視窗的用途

| 視窗 | 長度 | 用途 |
|---|---|---|
| 短期（1 小時） | 12 intervals | 偵測突發事件（DDoS、硬體故障） |
| 長期（1 天） | 288 intervals | 偵測緩慢漂移（設備老化、配置問題）|

短窗口反應快但容易被局部峰值干擾；長窗口穩健但對剛開始的事件反應慢。

### 為什麼要逐 port 獨立計算？

```
Port A 流量：  100 MB/s   ┐
Port B 流量：    1 MB/s   ┘ 若混在一起算
                             → 平均 ~50 MB/s
                             → Port B 的任何值都「嚴重偏低」
                             → 誤報
```

每個 port 只和自己的歷史比，才有意義。

📎 延伸閱讀：[Moving average — Wikipedia](https://en.wikipedia.org/wiki/Moving_average)（有動態圖示）


## Step 4 — Rolling statistics 建立短期 baseline

`rolling` 的意思是「每一個時間點都往回看最近一段Time的資料」。\
例如 `rolling(12)` 在每個 5 分鐘點往回取 1 小時（12 個間隔）。

我們計算兩種 rolling 統計：

| 統計量 | 視窗 | 用途 |
|---|---|---|
| `mean_1h`, `std_1h` | 12 intervals (1 小時) | 短期 z-score 偵測 |
| `median_1d`, `mad_1d`, `p95_1d` | 288 intervals (1 天) | 較穩健的長期 baseline |

因為每個 port 的流量規模不同，計算時要**逐 port 獨立計算**，
才不會讓小流量 port 的異常被大流量 port 的數值淹沒。


In [ ]:
WATCH_COLS = ["traffic_total", "error_rate", "discard_rate",
              "broadcast_total", "avg_packet_size"]

def rolling_mad(x):
    med = np.median(x)
    return np.median(np.abs(x - med))

def add_rolling(group):
    g = group.copy().sort_values("timestamp")
    if "port_id" not in g.columns:
        g["port_id"] = group.name
    for col in WATCH_COLS:
        s = g[col].astype(float)
        g[f"{col}__mean_1h"]   = s.rolling(12,  min_periods=6).mean()
        g[f"{col}__std_1h"]    = s.rolling(12,  min_periods=6).std().fillna(0)
        g[f"{col}__median_1d"] = s.rolling(288, min_periods=72).median()
        g[f"{col}__mad_1d"]    = s.rolling(288, min_periods=72).apply(
            rolling_mad, raw=True)
        g[f"{col}__p95_1d"]    = s.rolling(288, min_periods=72).quantile(0.95)
    return g

print("計算 rolling statistics（逐 port，約需 20–40 秒）…")
df = df.groupby("port_id", group_keys=False).apply(add_rolling)
rolling_cols = [c for c in df.columns if "__" in c]
print(f"新增 {len(rolling_cols)} 個 rolling 欄位")


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5 — 疊加圖：原始值 vs rolling baseline

這張圖是本 lab 的核心視覺化。你會同時看到：

- 灰色細線 — 每 5 分鐘的原始流量
- 藍色粗線 — rolling mean（1 小時）
- 藍色陰影 — mean ± 3σ 的「正常區間」
- 紅色陰影 — 已知事件時段

如果某個時間點的原始值超出陰影帶，就是後續 Lab 02 要標記為異常的位置。


In [ ]:
PORT = "port-id7427"
COL  = "traffic_total"

p = df[df["port_id"] == PORT].copy().sort_values("timestamp")

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(p["timestamp"], p[COL],
        color="lightgray", lw=0.8, alpha=0.9, label="raw")
ax.plot(p["timestamp"], p[f"{COL}__mean_1h"],
        color="steelblue", lw=1.6, label="mean_1h")
ax.fill_between(
    p["timestamp"],
    p[f"{COL}__mean_1h"] - 3 * p[f"{COL}__std_1h"],
    p[f"{COL}__mean_1h"] + 3 * p[f"{COL}__std_1h"],
    alpha=0.15, color="steelblue", label="mean ± 3σ"
)
shade_events(ax, PORT, events)

ax.set_title(f"{PORT} — {COL}: raw value vs rolling baseline (red shading = event)")
ax.set_ylabel("Bytes / 5 min")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 6 — 儲存 features.csv

把所有特徵寫出去，讓 Lab 02–7 直接讀取，不需重新計算。


In [ ]:
out_path = DATA_PROCESSED / "features.csv"
df.to_csv(out_path, index=False)
print(f"✅ 儲存完成：{out_path}")
print(f"   筆數：{len(df):,}  |  欄位數：{df.shape[1]}")


---
💬 **討論暫停 ▸ 全班討論** — 停止執行，分享你的觀察。

---

## ⚠ 人類驗證點 #1 — 特徵設計是你的決定，不是 AI 的

本 lab 裡的特徵（`traffic_total`、`error_rate`、rolling 視窗 大小）是由你設計的，
不是演算法自動選出來的。這很重要：

**AI 可以計算任何你告訴它的特徵，但它不知道哪些特徵對你的網路有意義。**

在繼續之前，問自己：

1. 你選的 rolling 視窗（1 小時 / 1 天）適合你的業務節奏嗎？\
   24/7 的資料中心和朝九晚五的辦公室網路，baseline 的形狀完全不同。

2. `error_rate = errors / ucast_packets` 這個公式合理嗎？\
   如果某個 port 幾乎沒有流量（ucast 接近 0），error_rate 會發生什麼？

3. Step 3 的視覺化圖裡，哪個 port 的行為最特別？為什麼？

這些判斷需要你，不需要 AI。


---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習：Rolling 視窗 大小的影響

**任務**：將 Step 4 的短期視窗從 `12`（1 小時）改成 `6`（30 分鐘）。

```python
# 在 Step 4 的 code cell 中找到這行：
WATCH_COLS_SHORT = 12   # 或 rolling(12)
# 改成：
WATCH_COLS_SHORT = 6
```

重新執行 Step 4 和 Step 5，觀察：
- rolling mean 的曲線是否變得更「抖動」？
- 陰影帶（mean ± 3σ）是否變窄了？
- 已知事件時段的異常是否更明顯，還是更多假陽性出現？

**討論**：視窗越短，偵測越敏感，但誤報也越多。你的業務環境偏向哪一側？

---

> 「`safe_div()` 解決了什麼問題？如果不用它，哪個特徵計算最危險？」

> 「為什麼 `traffic_asymmetry` 可能在半夜出現極端值，即使網路完全正常？」

> 「如果你的網路有明顯的上班 / 下班流量模式，1 小時視窗和 1 天視窗各有什麼問題？」

## 練習與檢查點

完成本 lab 後，你應該能回答三個問題：

1. 把 `PORT` 改成 `port-id7430`，重新執行 Step 3 的三張圖。\
   你觀察到哪種事件？哪個 panel 最明顯？

2. `avg_packet_size` 升高通常代表什麼業務行為？\
   （提示：大量小封包 vs 少量大封包的差異）

3. 為什麼 rolling statistics 必須「逐 port 獨立計算」，\
   而不能把所有 port 混在一起計算？

---

**下一步 → Lab 02 — Baseline 異常偵測**：\
讀取 `outputs/self-study/features.csv`，用 rolling baseline 標記異常時間點。
